#HackerRank Problem - Advance Certificate
##Suspicious Criteria
- 2 or more transactions from the same sender
- Each transaction in the sequence occurs within 1 hour of the previous one
- The total amount in that sequence >= 150

In [0]:
%sql
use biki;

# Setup sample data for SQL table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id INT PRIMARY KEY,
    sender_id INT NOT NULL,
    receiver_id INT NOT NULL,
    transaction_time TIMESTAMP NOT NULL,
    amount DECIMAL(10,2) NOT NULL
);

-- INSERT INTO transactions VALUES
-- -- Case 1: Valid suspicious sequence (exactly >=150 within 1 hour)
-- (1, 201, 301, '2026-03-12 09:00:00', 70),
-- (2, 201, 302, '2026-03-12 09:20:00', 40),
-- (3, 201, 303, '2026-03-12 09:50:00', 40),
-- -- Case 2: Gap > 1 hour breaks sequence
-- (4, 202, 304, '2026-03-12 10:00:00', 100),
-- (5, 202, 305, '2026-03-12 11:05:00', 60),
-- -- Case 3: Only 1 transaction (should not count)
-- (6, 203, 306, '2026-03-12 10:30:00', 200),
-- -- Case 4: Within 1 hour but total <150
-- (7, 204, 307, '2026-03-12 11:00:00', 50),
-- (8, 204, 308, '2026-03-12 11:20:00', 40),
-- (9, 204, 309, '2026-03-12 11:50:00', 30),
-- -- Case 5: Exactly 1 hour gap (boundary condition)
-- (10, 205, 310, '2026-03-12 12:00:00', 80),
-- (11, 205, 311, '2026-03-12 13:00:00', 70),
-- -- Case 6: Overlapping sequences
-- (12, 206, 312, '2026-03-12 14:00:00', 60),
-- (13, 206, 313, '2026-03-12 14:20:00', 60),
-- (14, 206, 314, '2026-03-12 14:40:00', 40),
-- (15, 206, 315, '2026-03-12 15:10:00', 30),
-- -- Case 7: Multiple sequences from same sender
-- (16, 207, 316, '2026-03-12 16:00:00', 90),
-- (17, 207, 317, '2026-03-12 16:30:00', 70),
-- (18, 207, 318, '2026-03-12 18:00:00', 100),
-- (19, 207, 319, '2026-03-12 18:20:00', 60);

In [0]:
%sql
select * from transactions;

In [0]:
%sql
with prev_transc_time as (
select *
  , lag(transaction_time) over(partition by sender_id order by transaction_time) as prev_transaction_time 
from transactions
), new_group as (
select *
  , timestampdiff(minute ,prev_transaction_time, transaction_time)
  , case when prev_transaction_time is null or timestampdiff(minute ,prev_transaction_time, transaction_time)>60 then 1
    else 0 end as new_group
from prev_transc_time
), grouped_transc as (
select *
  , sum(new_group) over(partition by sender_id order by transaction_time) as group_id
from new_group
) select sender_id
  , min(transaction_time) as start_time
  , max(transaction_time) as end_time
  , count(1) as transaction_count
  , sum(amount) as total_amount
from grouped_transc group by sender_id, group_id
having count(1)>=2 and sum(amount)>=150 
order by sender_id, start_time, end_time;

# Setup Sample dataframe for PySpark

In [0]:
from pyspark.sql import SparkSession, types as T, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, FloatType, TimestampType
from pyspark.sql.window import Window as W
from pyspark.conf import SparkConf

In [0]:
conf=SparkConf().setAppName('HackerRank')
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [0]:
data=[# Case 1: Valid suspicious sequence (exactly >=150 within 1 hour)
(1, 201, 301, '2026-03-12 09:00:00', 70),
(2, 201, 302, '2026-03-12 09:20:00', 40),
(3, 201, 303, '2026-03-12 09:50:00', 40),
# Case 2: Gap > 1 hour breaks sequence
(4, 202, 304, '2026-03-12 10:00:00', 100),
(5, 202, 305, '2026-03-12 11:05:00', 60),
# Case 3: Only 1 transaction (should not count)
(6, 203, 306, '2026-03-12 10:30:00', 200),
# Case 4: Within 1 hour but total <150
(7, 204, 307, '2026-03-12 11:00:00', 50),
(8, 204, 308, '2026-03-12 11:20:00', 40),
(9, 204, 309, '2026-03-12 11:50:00', 30),
# Case 5: Exactly 1 hour gap (boundary condition)
(10, 205, 310, '2026-03-12 12:00:00', 80),
(11, 205, 311, '2026-03-12 13:00:00', 70),
# Case 6: Overlapping sequences
(12, 206, 312, '2026-03-12 14:00:00', 60),
(13, 206, 313, '2026-03-12 14:20:00', 60),
(14, 206, 314, '2026-03-12 14:40:00', 40),
(15, 206, 315, '2026-03-12 15:10:00', 30),
# Case 7: Multiple sequences from same sender
(16, 207, 316, '2026-03-12 16:00:00', 90),
(17, 207, 317, '2026-03-12 16:30:00', 70),
(18, 207, 318, '2026-03-12 18:00:00', 100),
(19, 207, 319, '2026-03-12 18:20:00', 60)]

In [0]:
# columns=['transaction_id','sender_id','receiver_id','transaction_time','amount']
columns = StructType([StructField('transaction_id', IntegerType(), True)
            , StructField('sender_id', IntegerType(), True)
            , StructField('receiver_id', IntegerType(), True)
            , StructField('transaction_time', StringType(), True)
            , StructField('amount', DoubleType(), True)
          ])

In [0]:
df=spark.createDataFrame(data, schema=columns)
df.show(truncate=False)

In [0]:
windw = W.partitionBy('sender_id').orderBy('transaction_time')

In [0]:
new_df=df.withColumn('transaction_time', F.to_timestamp('transaction_time')) \
    .withColumn('prev_time', F.lag('transaction_time').over(windw)) \
    .withColumn('time_diff', F.timestamp_diff('minute', 'prev_time', 'transaction_time')) \
    .withColumn('new_group', F.when((F.col('time_diff')>60)|(F.col('prev_time').isNull()),F.lit(1)).otherwise(F.lit(0)))

grouped_df=new_df.withColumn('group_id', F.sum('new_group').over(windw))
grouped_df.show(truncate=False)

In [0]:
result_df=grouped_df.groupBy(['sender_id', 'group_id'])\
        .agg(F.min('transaction_time').alias('min_transaction_time'), \
            F.max('transaction_time').alias('max_transaction_time'), \
            F.sum('amount').alias('total_amount'), \
            F.count('sender_id').alias('transaction_count')) \
        .filter((F.col('transaction_count')>=2) & (F.col('total_amount')>=150)) \
        .drop('group_id') \
        .orderBy('sender_id', 'group_id')
result_df.show(truncate=False)